# Importações

In [2]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re

# RF01 – Criar ou Carregar o Dataset de Vendas


In [ ]:
def gerar_dataset_vendas(n_registros=150, seed=42):
    """
    Gera um dataset sintético de vendas com dados propositalmente sujos,
    incluindo valores nulos, strings sujas, datas inválidas e outliers.
    """

    random.seed(seed)
    np.random.seed(seed)

    produtos = [
        'Notebook',
        'Smartphone',
        'Tablet',
        'Monitor',
        'Teclado',
        'Mouse'
    ]

    precos = {
        'Notebook': 3500,
        'Smartphone': 2200,
        'Tablet': 1800,
        'Monitor': 1200,
        'Teclado': 250,
        'Mouse': 120
    }

    categorias = {
        "Notebook": "Computadores",
        "Smartphone": "Celulares",
        "Tablet": "Celulares",
        "Monitor": "Computadores",
        "Teclado": "Periféricos",
        "Mouse": "Periféricos"
    }

    regioes = [
        "Sudeste",
        "Sul",
        "Nordeste",
        "Centro-Oeste",
        "Norte"
    ]

    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]

    data_inicio = datetime(2024, 1, 1)

    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]

        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Inserindo dados intencionalmente sujos para limpeza
        if random.random() < 0.05:
            quantidade = None  # valor nulo

        if random.random() < 0.04:
            preco = None  # valor nulo

        if random.random() < 0.03:
            produto = " " + produto  # espaço extra (string suja)

        data_str = (
            data.strftime("%Y-%m-%d")
            if random.random() > 0.02
            else "DATA INVALIDA"
        )

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })

    return pd.DataFrame(dados)


# Gerar e salvar
df_bruto = gerar_dataset_vendas()
os.makedirs("../data/raw", exist_ok=True)
df_bruto.to_csv("../data/raw/vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")


In [ ]:
df_bruto

RF02 – Inspecionar e Descrever os Dados

In [7]:
df = df_bruto

In [ ]:
def inspecionar_dados(df):
  """Exibe informações básicas do df."""
  print("\n=== INSPEÇÃO INICIAL DO DATASET ===" )
  print(f"Shape: {df.shape}")
  print(f"\nColunas: {list(df.columns)}" )
  print(f"\nTipos de dados:\n{df.dtypes}" )
  print(f"\nValores nulos por coluna:\n{df.isnull().sum()}" )
  print(f"\nValores duplicados: {df.duplicated().sum()}")
  print(f"\nPrimeiros registros:\n{df.head()}" )
  print(f"\nÚltimos registros:\n{df.tail()}" )
  print(f"\nEstatísticas descritivas:\n{df.describe()}" )
  return df.describe(include="all")

inspecionar_dados(df)

1. O dataset possui 150 registros e 8 colunas;
2. Há colunas numéricas ('id_venda', 'quantidade', 'preco_unitario') e categóricas ('cliente', 'produto', 'categoria', 'regiao');
3. A coluna 'data_venda' está como object TEXTO e precisará ser convertida para datetime;
4. Foram encontrados valores nulos para as colunas 'quantidade' e 'preco_unitario'. Logo, precisarão ser tratados;
5. Não parece ter outliers significativos;
6. Não há registros duplicados.

In [ ]:
def inspecionar_dados_vendas(df):
    print("\n=== ANÁLISE ESPECÍFICA DO DATASET ===")

    print("\nColunas categóricas:")
    #print(df["cliente"].value_counts())
    print(f"{df["produto"].value_counts()}")
    print(f"\n{df["categoria"].value_counts()}")
    print(f"\n{df["regiao"].value_counts()}")

    print("\nDatas inválidas:")
    print(df[df["data_venda"] == "DATA INVALIDA"])

    print("\nProdutos com espaços extras:")
    print(df[df["produto"] != df["produto"].str.strip()])

inspecionar_dados_vendas(df)

1. Há datas inválidas: DATA INVALIDA;
2. Coluna 'produto' possui valores com espaços em branco.

In [ ]:
df

Será necessário:

1. Tratar valores nulos para as colunas 'quantidade' e 'preco_unitario';
2. Remover linhas com datas inválidas: DATA INVÁLIDA;
3. Remover espaços em branco das strings para a coluna 'produto';
4. Converter 'data_venda' para o tipo datetime.

RF03 – Limpar e Tratar os Dados

In [ ]:
def remover_espacos_branco(df):
    """
    Remove espaços em branco extras de colunas do tipo texto
    """
    df = df.copy()

    padrao_espacos = re.compile(r"\s+")

    colunas_texto = df.select_dtypes(include="object").columns

    for coluna in colunas_texto:
        df[coluna] = df[coluna].apply(
            lambda texto:
                padrao_espacos.sub(" ", str(texto)).strip()
                if pd.notna(texto) else texto
        )

    return df


def converter_datas(df, colunas):
    """
    Converte colunas para datetime e remove linhas com datas inválidas
    """
    df = df.copy()

    qtd_registros_removidos = 0

    for coluna in colunas:
        df[coluna] = pd.to_datetime(df[coluna], errors="coerce")
        qtd_registros_removidos += df[coluna].isna().sum()

    df = df.dropna(subset=colunas)

    return df, int(qtd_registros_removidos)


def remover_linhas_nulas(df, colunas):
    """
    Remove linhas que possuem nulos nas colunas informadas.
    """
    df = df.copy()
    tamanho_df_antes = len(df)
    df = df.dropna(subset=colunas)
    qtd_registros_removidos = tamanho_df_antes - len(df)

    return df, qtd_registros_removidos


def converter_tipos(df, tipos):
    """
    Converte colunas para os tipos informados.

    Exemplo:
      {"quantidade": int, "preco_unitario": float}
    """
    df = df.copy()
    return df.astype(tipos)


def remover_colunas(df, colunas):
    """
    Remove coluna(s) de um df
    """

    df = df.copy()

    qtd_removidas = len([col for col in colunas if col in df.columns])
    df = df.drop(columns=colunas, errors="ignore")

    return df, qtd_removidas


def gerar_relatorio(titulo, **metricas):
    """
    Gera relatório da limpeza
    """
    relatorio = {}
    relatorio.update(metricas)

    # Exibe relatório
    print(f"\n=== RELATÓRIO - {titulo} ===")
    for chave, valor in relatorio.items():
        print(f"{chave}: {valor}")

    return relatorio


def limpar_dados(df):
    df = remover_espacos_branco(df.copy())

    qtd_inicial = len(df)

    df, qtd_datas_invalidas = converter_datas(df, ["data_venda"])
    df, qtd_nulos_removidos = remover_linhas_nulas(df, ["quantidade", "preco_unitario"])
    df = converter_tipos(df, {"quantidade": int, "preco_unitario": float})
    #df, qtd_colunas_removidas = remover_colunas(df, ["id_venda"])

    relatorio = gerar_relatorio(
        "LIMPEZA DE DADOS",
        qtd_registros_iniciais=qtd_inicial,
        qtd_registros_finais=len(df),
        qtd_registros_removidos_total = qtd_inicial - len(df),
        qtd_datas_invalidas_removidas = qtd_datas_invalidas,
        qtd_linhas_nulas_removidas = qtd_nulos_removidos,
        #qtd_colunas_removidas = qtd_colunas_removidas
    )

    return df, relatorio

def gravar_df_csv(df, diretorio, nome_arquivo):
    """
    Salva o df em CSV.
    """
    os.makedirs(diretorio, exist_ok=True)
    caminho_arquivo = os.path.join(diretorio, nome_arquivo)
    df.to_csv(caminho_arquivo, index=False)
    print(f"\nArquivo salvo em: {caminho_arquivo}")


df_v1, relatorio = limpar_dados(df)
gravar_df_csv(df_v1, "../data/processed/v1_com_outliers", "vendas_v1.csv")


In [ ]:
df_v1 # Com outliers

In [ ]:
# Conferência do tratamento e limpeza
inspecionar_dados(df_v1)
inspecionar_dados_vendas(df_v1)

A remoção de linhas com valores nulos e datas inválidas não tem impacto significativo no dataset já que são pouquíssimas linhas.